# Gemma 4 + `llama-server` on Kaggle — simple launch notebook

This notebook is the minimal runtime launcher:

- guarded against accidental auto-execution
- auto-discovers the runtime artifact
- auto-discovers a GGUF model
- copies the runtime into `/kaggle/working`
- validates `ldd` and `--version`
- starts and stops the server
- includes minimal text chat helpers

No rebuild or packaging steps are included here.

Two runtime profiles are supported:
- `gpu` — expects the CUDA Kaggle dataset and shows GPU memory usage
- `cpu` — expects a separate CPU-only Kaggle dataset and shows RAM usage

Keep the CPU artifact in a different Kaggle dataset from the GPU artifact.


In [ ]:
RUN_NOTEBOOK = False

if not RUN_NOTEBOOK:
    raise SystemExit(
        "Execution is locked. Set RUN_NOTEBOOK = True in this first cell, then rerun the notebook from the top."
    )


In [ ]:
from pathlib import Path
import os
import json
import time
import shutil
import signal
import socket
import zipfile
import subprocess

SERVER_PROFILE = 'gpu'  # 'gpu' | 'cpu'

PROFILE_SETTINGS = {
    'gpu': {
        'artifact_dataset_root': Path('/kaggle/input/datasets/alexandreev/llama-server-cuda-build-kaggle'),
        'n_gpu_layers': 'all',
        'flash_attn': 'on',
        'use_cuda_runtime': True,
        'monitor_kind': 'gpu',
    },
    'cpu': {
        'artifact_dataset_root': Path('/kaggle/input/datasets/alexandreev/llama-server-cpu-build-kaggle'),
        'n_gpu_layers': 0,
        'flash_attn': 'off',
        'use_cuda_runtime': False,
        'monitor_kind': 'ram',
    },
}
if SERVER_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f'Unsupported SERVER_PROFILE: {SERVER_PROFILE!r}')

PROFILE = PROFILE_SETTINGS[SERVER_PROFILE]
ARTIFACT_DATASET_ROOT = PROFILE['artifact_dataset_root']
MODEL_SEARCH_ROOTS = [
    Path('/kaggle/input/models/alexandreev/gemma-4-26b-a4b-it-gguf/gguf/gemma-4-26b-a4b-it-ud-q6_k.gguf/1'),
    Path('/kaggle/input/models'),
]

WORK_ROOT = Path('/kaggle/working/llama_server_kaggle_simple')
RUNTIME_ROOT = WORK_ROOT / 'runtime'
LOG_DIR = WORK_ROOT / 'logs'
RUN_DIR = WORK_ROOT / 'run'
for p in [WORK_ROOT, RUNTIME_ROOT, LOG_DIR, RUN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HOST = '127.0.0.1'
PORT = 18081
CTX_SIZE = 2048
BATCH_SIZE = CTX_SIZE
UBATCH_SIZE = 512
PARALLEL = 1
N_GPU_LAYERS = PROFILE['n_gpu_layers']
FLASH_ATTN = PROFILE['flash_attn']
USE_CUDA_RUNTIME = PROFILE['use_cuda_runtime']
MONITOR_KIND = PROFILE['monitor_kind']
EXTRA_SERVER_ARGS = [
    '--cache-type-k', 'q4_0',
    '--cache-type-v', 'q4_0',
    '--reasoning', 'off',
    '--reasoning-budget', '0',
    '--reasoning-format', 'none',
]
PID_FILE = RUN_DIR / 'llama-server.pid'
LOG_FILE = LOG_DIR / 'llama-server.log'
BINARY_PATH = RUNTIME_ROOT / 'llama-server'

print('SERVER_PROFILE =', SERVER_PROFILE)
print('ARTIFACT_DATASET_ROOT =', ARTIFACT_DATASET_ROOT)


In [ ]:
# BOOTSTRAP CELL
# After a Kaggle VM reboot, rerun this cell first. It restores core paths,
# reuses an already staged runtime in /kaggle/working when possible, otherwise
# copies it again from the attached dataset.

from pathlib import Path
import os
import json
import shutil
import zipfile

PROFILE_SETTINGS = globals().get('PROFILE_SETTINGS', {
    'gpu': {
        'artifact_dataset_root': Path('/kaggle/input/datasets/alexandreev/llama-server-cuda-build-kaggle'),
        'n_gpu_layers': 'all',
        'flash_attn': 'on',
        'use_cuda_runtime': True,
        'monitor_kind': 'gpu',
    },
    'cpu': {
        'artifact_dataset_root': Path('/kaggle/input/datasets/alexandreev/llama-server-cpu-build-kaggle'),
        'n_gpu_layers': 0,
        'flash_attn': 'off',
        'use_cuda_runtime': False,
        'monitor_kind': 'ram',
    },
})
SERVER_PROFILE = globals().get('SERVER_PROFILE', 'gpu')
if SERVER_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f'Unsupported SERVER_PROFILE: {SERVER_PROFILE!r}')
PROFILE = PROFILE_SETTINGS[SERVER_PROFILE]

ARTIFACT_DATASET_ROOT = globals().get('ARTIFACT_DATASET_ROOT', PROFILE['artifact_dataset_root'])
MODEL_SEARCH_ROOTS = globals().get('MODEL_SEARCH_ROOTS', [
    Path('/kaggle/input/models/alexandreev/gemma-4-26b-a4b-it-gguf/gguf/gemma-4-26b-a4b-it-ud-q6_k.gguf/1'),
    Path('/kaggle/input/models'),
])

WORK_ROOT = globals().get('WORK_ROOT', Path('/kaggle/working/llama_server_kaggle_simple'))
RUNTIME_ROOT = globals().get('RUNTIME_ROOT', WORK_ROOT / 'runtime')
LOG_DIR = globals().get('LOG_DIR', WORK_ROOT / 'logs')
RUN_DIR = globals().get('RUN_DIR', WORK_ROOT / 'run')
for p in [WORK_ROOT, RUNTIME_ROOT, LOG_DIR, RUN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HOST = globals().get('HOST', '127.0.0.1')
PORT = globals().get('PORT', 18081)
CTX_SIZE = globals().get('CTX_SIZE', 2048)
BATCH_SIZE = globals().get('BATCH_SIZE', CTX_SIZE)
UBATCH_SIZE = globals().get('UBATCH_SIZE', 512)
PARALLEL = globals().get('PARALLEL', 1)
N_GPU_LAYERS = globals().get('N_GPU_LAYERS', PROFILE['n_gpu_layers'])
FLASH_ATTN = globals().get('FLASH_ATTN', PROFILE['flash_attn'])
USE_CUDA_RUNTIME = globals().get('USE_CUDA_RUNTIME', PROFILE['use_cuda_runtime'])
MONITOR_KIND = globals().get('MONITOR_KIND', PROFILE['monitor_kind'])
EXTRA_SERVER_ARGS = globals().get('EXTRA_SERVER_ARGS', [
    '--cache-type-k', 'q4_0',
    '--cache-type-v', 'q4_0',
    '--reasoning', 'off',
    '--reasoning-budget', '0',
    '--reasoning-format', 'none',
])
PID_FILE = globals().get('PID_FILE', RUN_DIR / 'llama-server.pid')
LOG_FILE = globals().get('LOG_FILE', LOG_DIR / 'llama-server.log')

def discover_model(search_roots: list[Path]) -> Path:
    for root in search_roots:
        if root.is_file() and root.suffix == '.gguf':
            return root
        if root.exists() and root.is_dir():
            direct = sorted([p for p in root.glob('*.gguf') if p.is_file()])
            if direct:
                return direct[0]
    for root in search_roots:
        if root.exists() and root.is_dir():
            hits = sorted([p for p in root.rglob('*.gguf') if p.is_file()])
            if hits:
                return hits[0]
    raise SystemExit('No GGUF model found. Attach a model dataset in Kaggle Add Input, then rerun.')

def _copy_runtime_layout(dataset_root: Path, out_root: Path) -> str:
    if (dataset_root / 'llama-server').exists():
        for p in dataset_root.iterdir():
            if p.is_file():
                shutil.copy2(p, out_root / p.name)
        return 'flat'
    if (dataset_root / 'bin' / 'llama-server').exists():
        for p in (dataset_root / 'bin').iterdir():
            if p.is_file():
                shutil.copy2(p, out_root / p.name)
        return 'bin_dir'
    if (dataset_root / 'runtime' / 'llama-server').exists():
        for p in (dataset_root / 'runtime').iterdir():
            if p.is_file():
                shutil.copy2(p, out_root / p.name)
        return 'runtime_dir'
    zip_files = sorted(dataset_root.glob('*.zip'))
    if zip_files:
        with zipfile.ZipFile(zip_files[0], 'r') as z:
            z.extractall(out_root)
        extracted_runtime = out_root / 'runtime'
        if (extracted_runtime / 'llama-server').exists():
            tmp = out_root / '__tmp__'
            tmp.mkdir(parents=True, exist_ok=True)
            for p in extracted_runtime.iterdir():
                if p.is_file():
                    shutil.move(str(p), str(tmp / p.name))
            shutil.rmtree(out_root)
            out_root.mkdir(parents=True, exist_ok=True)
            for p in tmp.iterdir():
                shutil.move(str(p), str(out_root / p.name))
            shutil.rmtree(tmp, ignore_errors=True)
        return 'zip'
    raise SystemExit(
        f'No supported runtime layout found under {dataset_root}. '
        f'For SERVER_PROFILE={SERVER_PROFILE!r}, attach the matching Kaggle dataset.'
    )

def prepare_runtime(dataset_root: Path, out_root: Path, force_restage: bool = False) -> dict:
    if (out_root / 'llama-server').exists() and not force_restage:
        return {'layout': 'working_copy', 'runtime_root': str(out_root)}
    if out_root.exists():
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    layout = _copy_runtime_layout(dataset_root, out_root)
    for maybe_exec in [out_root / 'llama-server', out_root / 'llama-server.sh', out_root / 'llama-cli', out_root / 'llama-cli.sh']:
        if maybe_exec.exists():
            maybe_exec.chmod(maybe_exec.stat().st_mode | 0o111)
    return {'layout': layout, 'runtime_root': str(out_root)}

MODEL_FILE = discover_model(MODEL_SEARCH_ROOTS)
runtime_info = prepare_runtime(ARTIFACT_DATASET_ROOT, RUNTIME_ROOT, force_restage=False)
BINARY_PATH = globals().get('BINARY_PATH', RUNTIME_ROOT / 'llama-server')
print(json.dumps({
    'server_profile': SERVER_PROFILE,
    'artifact_dataset_root': str(ARTIFACT_DATASET_ROOT),
    'runtime_root': str(RUNTIME_ROOT),
    'binary_path': str(BINARY_PATH),
    'model_file': str(MODEL_FILE),
    'runtime_source': runtime_info['layout'],
    'monitor_kind': MONITOR_KIND,
}, indent=2, ensure_ascii=False))


In [ ]:
def clean_env(runtime_root: Path, use_cuda_runtime: bool = USE_CUDA_RUNTIME) -> dict[str, str]:
    env = {
        'HOME': os.environ.get('HOME', '/kaggle/working'),
        'PATH': '/usr/bin:/bin',
        'LANG': 'C.UTF-8',
        'LC_ALL': 'C.UTF-8',
    }
    ld_parts = [str(runtime_root)]
    if use_cuda_runtime:
        for part in [
            '/usr/local/nvidia/lib64',
            '/usr/local/cuda/lib64',
            '/usr/local/cuda-12.8/lib64',
            '/usr/local/cuda-12.8/targets/x86_64-linux/lib',
        ]:
            if Path(part).exists():
                ld_parts.append(part)
    if os.environ.get('LD_LIBRARY_PATH'):
        ld_parts.append(os.environ['LD_LIBRARY_PATH'])
    env['LD_LIBRARY_PATH'] = ':'.join(ld_parts)
    return env

def _read_meminfo() -> dict[str, int]:
    data: dict[str, int] = {}
    meminfo = Path('/proc/meminfo')
    if not meminfo.exists():
        return data
    for line in meminfo.read_text().splitlines():
        if ':' not in line:
            continue
        key, raw = line.split(':', 1)
        parts = raw.strip().split()
        if parts and parts[0].isdigit():
            data[key] = int(parts[0])
    return data

def _kb_to_gib(value_kb: int | None) -> float | None:
    if value_kb is None:
        return None
    return round(value_kb / 1024 / 1024, 2)

def resource_snapshot(pid: int | None = None) -> dict:
    if MONITOR_KIND == 'gpu':
        proc = subprocess.run(
            ['bash', '-lc', 'nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits || true'],
            text=True,
            capture_output=True,
        )
        rows = []
        for line in (proc.stdout or '').splitlines():
            parts = [part.strip() for part in line.split(',')]
            if len(parts) >= 5:
                rows.append({
                    'index': parts[0],
                    'name': parts[1],
                    'memory_used_mib': parts[2],
                    'memory_total_mib': parts[3],
                    'utilization_gpu_pct': parts[4],
                })
        return {'monitor_kind': 'gpu', 'gpus': rows, 'stderr': (proc.stderr or '').strip()}

    info = _read_meminfo()
    total = info.get('MemTotal')
    available = info.get('MemAvailable')
    used = total - available if total is not None and available is not None else None
    return {
        'monitor_kind': 'ram',
        'mem_total_gib': _kb_to_gib(total),
        'mem_available_gib': _kb_to_gib(available),
        'mem_used_gib': _kb_to_gib(used),
    }

env = clean_env(RUNTIME_ROOT, use_cuda_runtime=USE_CUDA_RUNTIME)
print('==== profile ====')
print(SERVER_PROFILE)
print('==== ldd ====')
proc = subprocess.run(['ldd', str(RUNTIME_ROOT / 'llama-server')], text=True, capture_output=True, env=env)
print(proc.stdout or proc.stderr)
print('==== version ====')
wrapper = RUNTIME_ROOT / 'llama-server.sh'
binary = wrapper if wrapper.exists() else (RUNTIME_ROOT / 'llama-server')
proc = subprocess.run([str(binary), '--version'], text=True, capture_output=True, env=env)
print(proc.stdout or proc.stderr)
print('==== resources ====')
print(json.dumps(resource_snapshot(), indent=2, ensure_ascii=False))


In [ ]:
def read_pid() -> int | None:
    if PID_FILE.exists():
        try:
            return int(PID_FILE.read_text().strip())
        except Exception:
            return None
    return None

def process_exists(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def wait_for_pid_exit(pid: int, timeout_s: float = 20.0, poll_s: float = 0.5) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if not process_exists(pid):
            return True
        time.sleep(poll_s)
    return not process_exists(pid)

def tail_log(n: int = 120) -> str:
    if not LOG_FILE.exists():
        return ''
    lines = LOG_FILE.read_text(errors='ignore').splitlines()
    return '\n'.join(lines[-n:])

def is_port_open(host: str = HOST, port: int = PORT, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0

def build_server_command(*, runtime_root: Path = RUNTIME_ROOT, model_file: Path = MODEL_FILE, host: str = HOST, port: int = PORT, ctx_size: int = CTX_SIZE, batch_size: int | None = BATCH_SIZE, ubatch_size: int = UBATCH_SIZE, parallel: int = PARALLEL, n_gpu_layers: str | int = N_GPU_LAYERS, flash_attn: str = FLASH_ATTN, extra_args: list[str] | None = None) -> list[str]:
    if batch_size is None:
        batch_size = ctx_size
    wrapper = runtime_root / 'llama-server.sh'
    binary = wrapper if wrapper.exists() else (runtime_root / 'llama-server')
    resolved_extra_args = list(EXTRA_SERVER_ARGS if extra_args is None else extra_args)
    cmd = [
        str(binary), '-m', str(model_file), '--host', str(host), '--port', str(port),
        '--ctx-size', str(ctx_size), '--batch-size', str(batch_size), '--ubatch-size', str(ubatch_size),
        '--parallel', str(parallel), '--n-gpu-layers', str(n_gpu_layers), '--flash-attn', str(flash_attn),
    ]
    cmd.extend(str(x) for x in resolved_extra_args)
    return cmd

def stop_server(force: bool = False, timeout_s: float = 20.0) -> bool:
    pid = read_pid()
    if pid is None:
        PID_FILE.unlink(missing_ok=True)
        print('No PID file found.')
        return True
    if not process_exists(pid):
        PID_FILE.unlink(missing_ok=True)
        print(f'Removed stale PID file for PID {pid}.')
        return True
    sig = signal.SIGKILL if force else signal.SIGTERM
    os.kill(pid, sig)
    stopped = wait_for_pid_exit(pid, timeout_s=timeout_s)
    if stopped:
        PID_FILE.unlink(missing_ok=True)
        print(f'Stopped PID {pid} with {"SIGKILL" if force else "SIGTERM"}.')
        return True
    if force:
        raise RuntimeError(f'PID {pid} is still alive even after SIGKILL.')
    raise RuntimeError(f'PID {pid} did not exit after SIGTERM. Call stop_server(force=True) only if you explicitly want SIGKILL.')

def wait_for_server(host: str = HOST, port: int = PORT, timeout_s: int = 180) -> bool:
    import requests
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if LOG_FILE.exists():
            tail = tail_log(80)
            if 'failed to load model' in tail or 'error loading model' in tail:
                return False
        try:
            r = requests.get(f'http://{host}:{port}/health', timeout=3)
            if r.status_code < 500:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False

def start_server(*, runtime_root: Path = RUNTIME_ROOT, model_file: Path = MODEL_FILE, host: str = HOST, port: int = PORT, ctx_size: int = CTX_SIZE, batch_size: int | None = BATCH_SIZE, ubatch_size: int = UBATCH_SIZE, parallel: int = PARALLEL, n_gpu_layers: str | int = N_GPU_LAYERS, flash_attn: str = FLASH_ATTN, extra_args: list[str] | None = None, stop_existing: bool = True):
    if stop_existing:
        stop_server(force=False)
    cmd = build_server_command(
        runtime_root=runtime_root, model_file=model_file, host=host, port=port, ctx_size=ctx_size,
        batch_size=batch_size, ubatch_size=ubatch_size, parallel=parallel,
        n_gpu_layers=n_gpu_layers, flash_attn=flash_attn, extra_args=EXTRA_SERVER_ARGS if extra_args is None else extra_args
    )
    env = clean_env(runtime_root, use_cuda_runtime=USE_CUDA_RUNTIME)
    with open(LOG_FILE, 'w') as logf:
        proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    PID_FILE.write_text(str(proc.pid), encoding='utf-8')
    print('Started PID:', proc.pid)
    print('Monitor:', MONITOR_KIND)
    print('Command:')
    print(' '.join(cmd))
    return proc


In [ ]:
# Run these two lines to launch:
# proc = start_server()
# print('Server ready:', wait_for_server())


In [ ]:
import requests

def server_health():
    try:
        r = requests.get(f'http://{HOST}:{PORT}/health', timeout=5)
        return {'status_code': r.status_code, 'text': r.text}
    except Exception as e:
        return {'error': str(e)}

def server_models():
    for path in ['/v1/models', '/models']:
        try:
            r = requests.get(f'http://{HOST}:{PORT}{path}', timeout=10)
            if r.ok:
                return r.json()
        except Exception:
            pass
    return None

print('Profile:', SERVER_PROFILE)
print('Port open:', is_port_open())
print('Health:', server_health())
mods = server_models()
print('Models:', json.dumps(mods, indent=2, ensure_ascii=False)[:2000] if mods else None)
print('Resources:', json.dumps(resource_snapshot(read_pid()), indent=2, ensure_ascii=False))
print('Log tail:')
print(tail_log(80))


In [ ]:
# !pip install -q openai requests httpx

import json
import requests
import httpx
from openai import OpenAI

def make_client():
    return OpenAI(base_url=f'http://{HOST}:{PORT}/v1', api_key='sk-local', timeout=httpx.Timeout(300.0, connect=10.0, read=300.0, write=30.0), max_retries=0)

def resolve_model_id():
    data = server_models()
    if isinstance(data, dict) and data.get('data'):
        return data['data'][0]['id']
    return MODEL_FILE.name

def new_dialog(system: str | None = 'You are a concise and helpful assistant.') -> list[dict]:
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    return msgs

def append_user(messages: list[dict], text: str):
    messages.append({'role': 'user', 'content': text})

def append_assistant(messages: list[dict], text: str):
    messages.append({'role': 'assistant', 'content': text})

def _field(obj, name: str):
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj.get(name)
    return getattr(obj, name, None)

def _extract_text_like(value) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                text = item.get('text')
                if isinstance(text, str):
                    parts.append(text)
                content = item.get('content')
                if isinstance(content, str):
                    parts.append(content)
        return ''.join(parts)
    if isinstance(value, dict):
        for key in ('text', 'content'):
            part = value.get(key)
            if isinstance(part, str):
                return part
        return ''
    return str(value)

def _extract_message_text(message) -> str:
    for field_name in ('content', 'reasoning_content'):
        text = _extract_text_like(_field(message, field_name))
        if text:
            return text
    return ''

def chat_openai(messages: list[dict], max_tokens: int = 256, stream: bool = False, print_output: bool = True) -> str:
    client = make_client()
    model = resolve_model_id()
    if not stream:
        response = client.chat.completions.create(model=model, messages=messages, temperature=0.7, max_tokens=max_tokens, stream=False)
        text = _extract_message_text(response.choices[0].message)
        if print_output:
            print(text)
        append_assistant(messages, text)
        return text
    parts = []
    stream_resp = client.chat.completions.create(model=model, messages=messages, temperature=0.7, max_tokens=max_tokens, stream=True)
    for chunk in stream_resp:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        piece = _extract_message_text(delta)
        if piece:
            if print_output:
                print(piece, end='', flush=True)
            parts.append(piece)
    text = ''.join(parts)
    if print_output:
        print()
    append_assistant(messages, text)
    return text

def chat_requests(messages: list[dict], max_tokens: int = 256, stream: bool = False, print_output: bool = True) -> str:
    payload = {'model': resolve_model_id(), 'messages': messages, 'temperature': 0.7, 'max_tokens': max_tokens, 'stream': stream}
    url = f'http://{HOST}:{PORT}/v1/chat/completions'
    if not stream:
        r = requests.post(url, json=payload, timeout=300)
        r.raise_for_status()
        data = r.json()
        text = _extract_message_text(data['choices'][0].get('message', {}))
        if print_output:
            print(text)
        append_assistant(messages, text)
        return text
    r = requests.post(url, json=payload, timeout=300, stream=True)
    r.raise_for_status()
    parts = []
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue
        payload_str = line[6:]
        if payload_str == '[DONE]':
            break
        obj = json.loads(payload_str)
        if not obj.get('choices'):
            continue
        piece = _extract_message_text(obj['choices'][0].get('delta', {}))
        if piece:
            if print_output:
                print(piece, end='', flush=True)
            parts.append(piece)
    text = ''.join(parts)
    if print_output:
        print()
    append_assistant(messages, text)
    return text


In [ ]:
RUN_DEMO = False

if not RUN_DEMO:
    print("Set RUN_DEMO = True and run this cell to execute the text demo.")
else:
    messages = new_dialog('You are a concise assistant.')
    append_user(messages, 'Write a short sci-fi story on the Moon in 120 words or less.')
    text1 = chat_openai(messages, max_tokens=180, stream=False, print_output=True)
    print("\n--- returned from chat_openai ---")
    print(repr(text1[:300]))

    append_user(messages, 'Continue in two more sentences.')
    text2 = chat_requests(messages, max_tokens=80, stream=False, print_output=True)
    print("\n--- returned from chat_requests ---")
    print(repr(text2[:300]))


Notes:

- After a Kaggle VM reboot, rerun the **BOOTSTRAP CELL** and then the server-management cell.
- `stop_server()` is PID-file only. No `lsof`, no port-wide kill fallback.
- Text helpers now fall back to `message.reasoning_content` when `message.content` is empty.
- `EXTRA_SERVER_ARGS` is appended through `build_server_command()` and `start_server()`.
- `SERVER_PROFILE = 'gpu'` expects the CUDA runtime dataset and keeps GPU monitoring enabled.
- `SERVER_PROFILE = 'cpu'` expects a separate CPU-only runtime dataset and switches monitoring to RAM usage.
- Do not point the CPU profile at the GPU Kaggle dataset.
